# Démo 6 : servir un modèle avec TorchServe

Objectif : écrire un handler, empaqueter le modèle dans une archive `.mar`, démarrer TorchServe, l'interroger par HTTP, puis l'arrêter.

**Données** : Fashion-MNIST (Zalando Research, licence MIT), fourni par torchvision : 70 000 images de vêtements en niveaux de gris, 28 x 28 pixels, 10 classes.

**Ce que vous allez voir**

- écrire un handler, qui décrit ce que fait le serveur à chaque requête ;
- empaqueter le modèle dans une archive `.mar` et démarrer TorchServe ;
- interroger l'API par HTTP et lire sa description OpenAPI.

**Durée prévue** : 10 min.

*Les sorties enregistrées dans ce notebook viennent d'une exécution dans Google Colab sur GPU A100, avec la graine fixée. Sur un autre GPU ou sur CPU, certaines valeurs peuvent légèrement différer : PyTorch ne garantit pas des résultats identiques d'une plateforme à l'autre.*

## 0. Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

Dans Colab, seuls les paquets absents du runtime ou à une autre version sont téléchargés. Si torch ou torchvision venaient à être réinstallés, redémarrez la session (**Exécution > Redémarrer la session**) puis relancez le notebook.

## 1. Télécharger Fashion-MNIST

**On télécharge Fashion-MNIST une fois pour toutes dans le dossier `data`.**

Le code :
- `datasets.FashionMNIST("data", train=..., download=True)` : télécharge les deux jeux s'ils ne sont pas déjà
  là ;
- `%%capture` : masque la barre de progression.

In [ ]:
%%capture
from torchvision import datasets

datasets.FashionMNIST("data", train=True, download=True)
datasets.FashionMNIST("data", train=False, download=True)

Les fichiers sont dans le dossier `data` : les cellules suivantes les retrouvent sans rien retélécharger.

## 2. Dataset et DataLoader

**Les images passent par un Dataset et un DataLoader, avec les trois jeux habituels.**

Le Dataset donne une image et son label à la demande ; le DataLoader les regroupe en lots de 256, mélangés
à chaque époque pour l'entraînement. La validation suit l'entraînement époque par époque ; le test reste
de côté.

Le code :
- `datasets.FashionMNIST(..., transform=transforms.ToTensor())` : chaque image devient un tenseur (1, 28, 28),
  aux valeurs entre 0 et 1 ;
- `random_split(train_complet, [50000, 10000])` : 50 000 images pour entraîner, 10 000 pour valider ;
- `DataLoader(train_ds, batch_size=256, shuffle=True)` : des lots mélangés pour l'entraînement ; les
  DataLoaders de validation et de test gardent l'ordre ;
- `images_test` : le premier lot de test, 256 images, qui servira à comparer les sorties après chaque export.

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_complet = datasets.FashionMNIST("data", train=True, transform=transforms.ToTensor())
test_ds = datasets.FashionMNIST("data", train=False, transform=transforms.ToTensor())
torch.manual_seed(0)
train_ds, val_ds = random_split(train_complet, [50000, 10000])
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)
images_test, _ = next(iter(test_loader))
print(len(train_ds), len(val_ds), len(test_ds), images_test.shape)

50000 images d'entraînement, 10000 de validation, 10000 de test ; un lot contient 256 images de forme (1, 28, 28).

## 3. Un CNN

**Une classe qui construit un CNN de la largeur voulue.**

La même architecture servira à plusieurs mesures ; seule sa largeur (le nombre de canaux) change.

Le code :
- `bloc1` et `bloc2` : deux blocs convolution, ReLU, pooling, avec `largeur` canaux, puis deux fois plus ;
- `classifieur` : `nn.Flatten()`, puis deux couches linéaires jusqu'aux 10 scores ;
- `forward` : les données traversent `bloc1`, `bloc2`, puis `classifieur`.

In [ ]:
import torch.nn as nn


class CNN(nn.Module):
    def __init__(self, largeur=32):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(1, largeur, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(largeur, 2 * largeur, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifieur = nn.Sequential(
            nn.Flatten(), nn.Linear(2 * largeur * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.classifieur(self.bloc2(self.bloc1(x)))

Rien à afficher : la classe est définie.

## 4. Une époque d'entraînement (fourni)

**Une époque d'entraînement, lot par lot.**

Le code (à survoler) : pour chaque lot du DataLoader, images et labels vont sur le device, puis la boucle
habituelle : loss, `zero_grad()`, `backward()`, `step()`.

In [ ]:
def entrainer_une_epoque(model, optimizer):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(model(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

Rien à afficher : la fonction est définie.

## 5. Mesurer loss et accuracy (fourni)

**Une fonction qui mesure la loss et l'accuracy d'un modèle sur tout un DataLoader.**

Le code (à survoler) :
- `model.eval()` et `torch.no_grad()` : on mesure sans entraîner ;
- pour chaque lot, la loss (pondérée par la taille du lot) et le nombre de bonnes réponses ;
- renvoie la loss moyenne et l'accuracy.

In [ ]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            total_loss += loss_fn(scores, labels).item() * len(labels)
            correct += (scores.argmax(dim=1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

Rien à afficher : la fonction est définie.

## 6. Entraîner et exporter en TorchScript

**On entraîne un modèle une époque, puis on l'exporte en TorchScript pour TorchServe.**

Le code :
- une époque d'entraînement, mesurée sur l'entraînement et la validation, comme dans la démo 5 ;
- `model.cpu().eval()` : le modèle sur CPU, en mode évaluation ;
- `torch.jit.trace(...).save("fashion_traced.pt")` : le fichier que TorchServe chargera.

In [ ]:
torch.manual_seed(0)
model = CNN().to(device)
loss_fn = nn.CrossEntropyLoss()
entrainer_une_epoque(model, torch.optim.Adam(model.parameters(), lr=1e-3))
train_loss, train_acc = evaluate(model, train_loader, loss_fn)
val_loss, val_acc = evaluate(model, val_loader, loss_fn)
print(f"époque 1  loss {train_loss:.3f}  acc {train_acc:.3f}  val_loss {val_loss:.3f}  val_acc {val_acc:.3f}")
model = model.cpu().eval()
torch.jit.trace(model, torch.randn(1, 1, 28, 28)).save("fashion_traced.pt")
print("modèle exporté")

`fashion_traced.pt` est écrit : un modèle TorchScript, qui se charge sans le code Python de la classe. C'est ce qu'attend TorchServe.

## 7. Vérifier Java

**TorchServe est écrit en partie en Java : il faut une machine Java.**

Le code :
- `shutil.which("java")` : cherche la commande `java` ;
- `apt-get install ... openjdk-17-jre-headless` : l'installe si elle manque (dans Colab) ;
- `java -version` : affiche la version installée.

In [ ]:
import shutil

if shutil.which("java") is None:
    !apt-get install -y -qq openjdk-17-jre-headless > /dev/null
!java -version

Java est disponible (`openjdk version "21.0.12"`). Les lignes `warning` viennent de Java, qui ne lit pas les limites de mémoire du conteneur de Colab ; elles sont sans conséquence ici.

## 8. Le handler

**Le handler décrit ce que fait le serveur à chaque requête.**

Le client envoie une image PNG ; le modèle attend un tenseur normalisé. Le handler fait le lien :
il prépare les images reçues, puis transforme les scores du modèle en réponse lisible.

Le code :
- `%%writefile handler.py` : écrit le contenu de la cellule dans un fichier ;
- `class FashionHandler(BaseHandler)` : `BaseHandler` sait déjà charger le modèle et l'appeler ;
- `preprocess` : lit chaque image reçue et la convertit en tenseur de la bonne taille ;
- `postprocess` : applique un softmax (des scores vers des probabilités) et renvoie la classe la plus probable
  avec sa probabilité.

In [ ]:
%%writefile handler.py
import io

import torch
from PIL import Image
from torchvision.transforms import v2
from ts.torch_handler.base_handler import BaseHandler

CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
PREPARER = v2.Compose([v2.Grayscale(), v2.Resize((28, 28)), v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])


class FashionHandler(BaseHandler):
    def preprocess(self, data):
        images = [Image.open(io.BytesIO(row.get("data") or row.get("body"))) for row in data]
        return torch.stack([PREPARER(image) for image in images])

    def postprocess(self, output):
        probas = torch.softmax(output, dim=1)
        return [{"classe": CLASSES[int(p.argmax())], "proba": round(float(p.max()), 3)} for p in probas]

`handler.py` est écrit. Le serveur appelle `preprocess`, puis le modèle, puis `postprocess` : le handler garantit que le prétraitement en production est le même qu'à l'entraînement.

**Piège fréquent** : un prétraitement différent de celui de l'entraînement ; le modèle répond, mais se trompe.

## 9. Empaqueter le modèle

**`torch-model-archiver` réunit le modèle et le handler dans une archive `.mar`.**

Une seule archive contient tout ce qu'il faut pour servir le modèle ; on la dépose dans un dossier
que TorchServe lit au démarrage.

Le code :
- `--model-name fashion` : le nom sous lequel le modèle sera appelé dans l'URL ;
- `--serialized-file fashion_traced.pt --handler handler.py` : le modèle et son handler ;
- `--export-path model_store` : le dossier des archives ; `ls` vérifie que l'archive est là.

In [ ]:
!mkdir -p model_store
!torch-model-archiver --model-name fashion --version 1.0 --serialized-file fashion_traced.pt --handler handler.py --export-path model_store --force
!ls model_store

`fashion.mar` est dans `model_store` : une seule archive qui contient le modèle, le handler et leurs métadonnées.

## 10. Configurer TorchServe

**Un fichier de configuration fixe les ports et le nombre de workers.**

Dans Colab, le port 8080, celui de TorchServe par défaut, est déjà pris. Et par défaut TorchServe
lance un worker (un processus qui exécute le modèle) par GPU ou par coeur de CPU ; un seul suffit ici.

Le code :
- `inference_address` : l'adresse de l'API de prédiction, port 7080 ;
- `management_address`, `metrics_address` : l'API de gestion et les métriques ;
- `default_workers_per_model=1` : un seul worker.

In [ ]:
%%writefile config.properties
inference_address=http://127.0.0.1:7080
management_address=http://127.0.0.1:7081
metrics_address=http://127.0.0.1:7082
default_workers_per_model=1

`config.properties` est écrit : l'API de prédiction écoutera sur le port 7080, l'API de gestion sur le 7081, les métriques sur le 7082.

## 11. Démarrer TorchServe

**Le serveur démarre en arrière-plan ; on attend qu'il réponde.**

Le code :
- `torchserve --start --ncs --ts-config config.properties --model-store model_store --models fashion=fashion.mar` :
  démarre le serveur avec notre archive ;
- `--disable-token-auth` : désactive le jeton d'authentification, pour la démo seulement ;
- `< /dev/null` et `> torchserve.log 2>&1` : la cellule rend la main, et les messages du serveur vont dans un
  fichier ;
- la boucle interroge `/ping` toutes les 2 secondes, jusqu'à une réponse.

In [ ]:
import time

import requests

# < /dev/null : sans cela, la cellule attend l'arrêt du serveur
!torchserve --start --ncs --ts-config config.properties --model-store model_store --models fashion=fashion.mar --disable-token-auth < /dev/null > torchserve.log 2>&1
for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:7080/ping", timeout=2).status_code == 200:
            break
    except requests.ConnectionError:
        time.sleep(2)
print(requests.get("http://127.0.0.1:7080/ping").json())

`{'status': 'Healthy'}` : le serveur a démarré et le modèle est chargé.

## 12. Interroger l'API

**On envoie une image au serveur, comme le ferait une application.**

Le code :
- `image.save("exemple.png")` : une image de test, enregistrée en PNG ;
- `requests.post("http://127.0.0.1:7080/predictions/fashion", data=...)` : une requête HTTP POST sur la route
  de prédiction, avec l'image dans le corps ;
- `reponse.json()` : la réponse, au format JSON.

In [ ]:
test = datasets.FashionMNIST("data", train=False)
image, label = test[0]
image.save("exemple.png")
with open("exemple.png", "rb") as fichier:
    reponse = requests.post("http://127.0.0.1:7080/predictions/fashion", data=fichier.read())
print("vraie classe :", test.classes[label])
print("réponse      :", reponse.json())

La réponse est un JSON : `Ankle boot`, avec une probabilité de 0.866, et c'est la bonne classe. Le client envoie une image PNG ; tout le reste se passe dans le serveur.

**Essayez** : envoyez une autre image de test, par exemple `test[1]`.

## 13. La description de l'API

**Une requête OPTIONS renvoie la description de l'API au format OpenAPI.**

C'est le contrat d'interface du service : les routes, les paramètres, les réponses. Il permet à
d'autres équipes d'écrire un client sans lire le code.

Le code :
- `requests.options("http://127.0.0.1:7080/")` : la description complète ;
- `description["openapi"]` : la version de la norme ; `description["paths"]` : les routes.

In [ ]:
description = requests.options("http://127.0.0.1:7080/").json()
print(description["openapi"])
print(list(description["paths"])[:6])

La description suit la norme OpenAPI 3.0.1. On y retrouve `/ping`, la route `/predictions/{model_name}` utilisée plus haut, une variante par version du modèle, et deux routes compatibles avec d'autres protocoles de service (`/v1/...:predict`, `/v2/.../infer`).

**À retenir** : c'est ce document que l'on remet à l'équipe qui écrira le client.

## 14. Arrêter le serveur

**On arrête TorchServe à la fin de la démo.**

In [ ]:
!torchserve --stop

`TorchServe has stopped.` : le serveur est arrêté et ses ports sont libérés.

## À retenir de cette démo

- TorchServe sert un modèle derrière une API REST : archive `.mar` (modèle et handler), serveur, requêtes HTTP.
- Le handler garantit que le prétraitement en production est le même qu'à l'entraînement.
- La description OpenAPI est le contrat d'interface à donner aux équipes qui appellent le service.